# AirQuery: train a custom gesture (MediaPipe Model Maker)

Trains a gesture classifier on frames recorded in the app and exports
`gesture_recognizer.task`. Drop that file into `web/public/models/` and AirQuery
loads it instead of the stock model (the capture panel then says `model: custom`).

**Model Maker replaces the stock gesture classifier**, so the new model must also
know the gestures AirQuery already uses. Record every label the app's capture
panel lists (press **C** in the app):

| folder | used for |
| --- | --- |
| `none` | hands doing nothing in particular (required by Model Maker) |
| `Closed_Fist`, `Open_Palm`, `Pointing_Up`, `Thumb_Down`, `Victory`, `ILoveYou` | the existing gestures (clutch, reset, point, undo, talk, lasso) |
| `Rock` (🤘, or whatever `config.gesture.customExplainLabel` says) | the new gesture: hold it to ask "what's this?" |

Each recording downloads as `<label>.tar` (about 60 frames). Record each label two
or three times with different angles, distances and lighting. Runs on a free Colab
CPU runtime in a few minutes; a GPU runtime is faster.

In [ ]:
!pip install -q mediapipe-model-maker

## 1. Upload the recordings

Select all the `<label>.tar` files the app downloaded. Files with the same label
are merged.

In [ ]:
import pathlib
import tarfile

from google.colab import files

uploaded = files.upload()  # choose every <label>.tar
root = pathlib.Path("dataset")
root.mkdir(exist_ok=True)
for name in uploaded:
    with tarfile.open(name) as t:
        t.extractall(root, filter="data")

for folder in sorted(root.iterdir()):
    print(f"{folder.name:14s} {len(list(folder.glob('*.jpg'))):4d} frames")

## 2. (Optional) more "no gesture" examples

MediaPipe's sample dataset has a `none` class; adding it makes false triggers rarer.

In [ ]:
!wget -q https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip
!unzip -qo rps_data_sample.zip
!mkdir -p dataset/none && cp rps_data_sample/none/* dataset/none/

## 3. Train and evaluate

Frames where no hand is detected are skipped automatically.

In [ ]:
from mediapipe_model_maker import gesture_recognizer

data = gesture_recognizer.Dataset.from_folder(
    dirname="dataset",
    hparams=gesture_recognizer.HandDataPreprocessingParams(),
)
train_data, rest = data.split(0.8)
validation_data, test_data = rest.split(0.5)

options = gesture_recognizer.GestureRecognizerOptions(
    hparams=gesture_recognizer.HParams(export_dir="exported_model", epochs=30),
)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data, validation_data=validation_data, options=options
)
loss, accuracy = model.evaluate(test_data, batch_size=1)
print(f"test accuracy: {accuracy:.3f}  (labels: {data.label_names})")

## 4. Export

Save the file as `web/public/models/gesture_recognizer.task` in the repo and reload
the app. Then re-run the gesture study (press **G**): a new model needs new numbers.

In [ ]:
model.export_model()
files.download("exported_model/gesture_recognizer.task")